# Multi-Agent Sycophancy — Conversation Viewer

In [1]:
import json
import glob
import html as html_lib
import re
from IPython.display import display, HTML
import ipywidgets as widgets

In [2]:
# ── Agent colours ──────────────────────────────────────────────────────────
AGENT_STYLES = {
    "claude_haiku": {"bg": "#e8f4fd", "border": "#2196F3", "label": "Claude Haiku",  "emoji": "🔵"},
    "gpt4o_mini":   {"bg": "#e8f8e8", "border": "#4CAF50", "label": "GPT-4o mini",   "emoji": "🟢"},
    "gemini_flash": {"bg": "#fff3e0", "border": "#FF9800", "label": "Gemini Flash",  "emoji": "🟠"},
    "user":         {"bg": "#f3e5f5", "border": "#9C27B0", "label": "User",           "emoji": "👤"},
}
DEFAULT_STYLE = {"bg": "#f5f5f5", "border": "#9e9e9e", "label": "Agent", "emoji": "🤖"}


def render_conversation(log_path: str) -> HTML:
    with open(log_path) as f:
        data = json.load(f)

    condition  = data.get("condition", "?")
    task       = data.get("task", "?")
    exp_id     = data.get("experiment_id", "")[:8]
    analysis   = data.get("analysis", {})

    # ── header ────────────────────────────────────────────────────────────
    condition_badge_color = "#e53935" if condition == "experimental" else "#1e88e5"
    out = f"""
    <div style='font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
                max-width: 860px; margin: 0 auto;'>
    <div style='background:#1a1a2e; color:white; padding:16px 20px; border-radius:10px 10px 0 0;
                display:flex; justify-content:space-between; align-items:center;'>
        <div>
            <span style='font-size:18px; font-weight:700;'>🧪 {task.replace("_"," ").title()}</span>
            <span style='margin-left:12px; background:{condition_badge_color}; padding:3px 10px;
                         border-radius:12px; font-size:12px; font-weight:600;'>{condition.upper()}</span>
        </div>
        <span style='font-size:11px; opacity:0.6;'>ID {exp_id}</span>
    </div>
    """

    # ── analysis banner ───────────────────────────────────────────────────
    accepted      = analysis.get("false_claim_accepted")
    verifications = analysis.get("verification_attempts", 0)
    first_doubt   = analysis.get("turns_until_questioning")
    markers       = ", ".join(analysis.get("linguistic_markers", [])) or "none"

    banner_color = "#ffebee" if accepted else "#e8f5e9"
    banner_icon  = "⚠️ False claim ACCEPTED" if accepted else "✅ False claim REJECTED"

    out += f"""
    <div style='background:{banner_color}; border:1px solid #ddd; padding:12px 20px;
                display:grid; grid-template-columns:1fr 1fr 1fr 1fr; gap:8px;'>
        <div><b>{banner_icon}</b></div>
        <div>🔍 Verification attempts: <b>{verifications}</b></div>
        <div>⏱ First doubt at turn: <b>{first_doubt if first_doubt else '—'}</b></div>
        <div>🏷 Markers: <b style='font-size:11px'>{markers}</b></div>
    </div>
    """

    # ── agent legend ──────────────────────────────────────────────────────
    agents_in_log = {e["agent"] for e in data.get("conversation", [])}
    legend_items  = ""
    for agent_name in sorted(agents_in_log):
        s = AGENT_STYLES.get(agent_name, DEFAULT_STYLE)
        legend_items += (
            f"<span style='background:{s['bg']}; border:2px solid {s['border']};"
            f" border-radius:8px; padding:4px 10px; margin-right:8px; font-size:13px;'>"
            f"{s['emoji']} {s['label']}</span>"
        )

    out += f"<div style='padding:10px 20px; background:#fafafa; border:1px solid #eee;'>{legend_items}</div>"

    # ── opening prompt ────────────────────────────────────────────────────
    opening = data.get("opening_prompt", "")
    if opening:
        out += f"""
        <div style='margin:16px 20px 4px; padding:12px 16px;
                    border-left:4px solid #9C27B0; background:#f3e5f5; border-radius:0 8px 8px 0;'>
            <div style='font-weight:700; font-size:14px; margin-bottom:8px;'>👤 User — Initial Prompt</div>
            <pre style='font-size:13px; line-height:1.6; white-space:pre-wrap; margin:0;'>{html_lib.escape(opening)}</pre>
        </div>"""

    # ── conversation turns ────────────────────────────────────────────────
    from config import TASKS, CONSENSUS_TOKEN
    false_claim_topic = TASKS.get(task, {}).get("false_claim_topic", "")

    current_turn = None
    for entry in data.get("conversation", []):
        turn       = entry.get("turn", 0)
        agent      = entry.get("agent", "?")
        response   = html_lib.escape(entry.get("response") or "")
        thinking   = entry.get("thinking")
        tokens_in  = entry.get("metadata", {}).get("tokens_input", 0)
        tokens_out = entry.get("metadata", {}).get("tokens_output", 0)
        model      = entry.get("metadata", {}).get("model", "")

        if not response:
            response = "<em style='color:#aaa'>(empty)</em>"

        # Turn divider
        if turn != current_turn:
            current_turn = turn
            out += f"""
            <div style='text-align:center; margin:16px 0 8px;
                        color:#888; font-size:12px; letter-spacing:1px;'>
                ── TURN {turn} ──
            </div>"""

        s = AGENT_STYLES.get(agent, DEFAULT_STYLE)

        # Highlight false claim topic
        if false_claim_topic:
            response = re.sub(
                f"({re.escape(html_lib.escape(false_claim_topic))})",
                r"<mark style='background:#ffeb3b; border-radius:3px;'>\1</mark>",
                response, flags=re.IGNORECASE,
            )

        # Highlight consensus token
        response = response.replace(
            html_lib.escape(CONSENSUS_TOKEN),
            f"<span style='background:#c8e6c9; border-radius:4px; padding:1px 6px;"
            f" font-size:11px; font-weight:700;'>{html_lib.escape(CONSENSUS_TOKEN)}</span>",
        )

        thinking_html = ""
        if thinking:
            thinking_html = f"""
            <details style='margin-top:8px;'>
                <summary style='cursor:pointer; color:#888; font-size:12px;'>💭 Show thinking</summary>
                <pre style='background:#f9f9f9; border-left:3px solid #ccc; padding:8px;
                            font-size:11px; white-space:pre-wrap; margin-top:6px;'>{html_lib.escape(thinking)}</pre>
            </details>"""

        out += f"""
        <div style='border-left:4px solid {s['border']}; background:{s['bg']};
                    margin:4px 20px; padding:12px 16px; border-radius:0 8px 8px 0;'>
            <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;'>
                <span style='font-weight:700; font-size:14px;'>{s['emoji']} {s['label']}</span>
                <span style='font-size:11px; color:#888;'>{model} &nbsp;·&nbsp; ↑{tokens_in} ↓{tokens_out} tokens</span>
            </div>
            <div style='font-size:14px; line-height:1.6; white-space:pre-wrap;'>{response}</div>
            {thinking_html}
        </div>"""

    out += "</div>"
    return HTML(out)


In [ ]:
# ── File picker ────────────────────────────────────────────────────────────
log_files = sorted(glob.glob("logs/*.json"))

if not log_files:
    display(HTML("<p style='color:red'>No log files found in logs/. Run an experiment first.</p>"))
else:
    dropdown = widgets.Dropdown(
        options=log_files,
        description="Log file:",
        layout=widgets.Layout(width="600px"),
        style={"description_width": "80px"},
    )
    out = widgets.Output()

    def on_change(change):
        with out:
            out.clear_output(wait=True)
            display(render_conversation(change["new"]))

    dropdown.observe(on_change, names="value")
    display(dropdown, out)

    # Render the first file immediately
    with out:
        display(render_conversation(log_files[0]))

Dropdown(description='Log file:', layout=Layout(width='600px'), options=('logs/binary_search_baseline_2026-03-…

Output()